# Power Outage Data Analysis (v2)

**Author (this version)**: Sonali Singh

**Original joint project**: Susana Haing, Sonali Singh — [shaing04/power-outages-analysis](https://github.com/shaing04/power-outages-analysis)

See `README.md` for the full attribution note and changelog. In short: the Introduction, Data Cleaning/EDA, Assessment of Missingness, and Hypothesis Testing sections below are carried over from the original joint DSC 80 project (Steps 1-4). Framing the Prediction Problem onward (Steps 5-8) is an independent rework done solely for this repository.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

import plotly.express as px
pd.options.plotting.backend = 'plotly'

import us
from us import states

from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)

## Step 1: Introduction *(joint work — carried over from the original project)*

In this project, we examine Purdue University's Laboratory for Advancing Sustainable Critical Infrastructure [data](https://engineering.purdue.edu/LASCI/research-data/outages) on major power outages in the continental United States from January 2000 to July 2016.

The data includes information about each major power outage, including geographical location, climate/weather, outage cause, and the number of people affected.

Our overarching research question is: **how do different causes affect power outages, and can we predict how long one will last?** We first test whether outage cause (specifically severe weather) has a measurable effect on outage duration, then build a model to predict duration from features available around the time an outage begins. Being able to give utilities and customers a good estimate of restoration time — based on cause, region, season, and time of day — is useful for triage and communication during an outage.

For our analysis, we focus on the following variables:

| Variable | Description |
|---|---|
| YEAR | The year an outage occurred |
| MONTH | The month an outage occurred |
| U.S._STATE | The specific US State the power outage occurred in |
| NERC.REGION | North American Electric Reliability Corporation region of an outage |
| CLIMATE.REGION | The 9 NOAA-defined climate regions in the United States |
| OUTAGE.START.DATE / TIME | Date and time an outage started |
| OUTAGE.RESTORATION.DATE / TIME | Date and time an outage was resolved |
| CAUSE.CATEGORY | Cause of the power outage |
| CLIMATE.CATEGORY | Whether it was a cold, warm, or normal climate period |
| OUTAGE.DURATION | The duration of an outage, in minutes |
| CUSTOMERS.AFFECTED | The number of customers affected by an outage |
| POPDEN_URBAN | Population density of urban areas (# persons per square mile) |


## Step 2: Data Cleaning and Exploratory Data Analysis *(joint work)*

The original dataset comes with 57 variables; we keep only the columns above. The start/restoration date and time columns are combined into single datetime columns (`OUTAGE.START`, `OUTAGE.RESTORATION`).

In [2]:
outages_df = pd.read_csv('outage.csv', header=5)

columns_to_keep = [
    'YEAR', 'MONTH', 'U.S._STATE', 'NERC.REGION', 'CLIMATE.REGION',
    'OUTAGE.START.DATE', 'OUTAGE.START.TIME',
    'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME',
    'CAUSE.CATEGORY', 'CLIMATE.CATEGORY', 'OUTAGE.DURATION',
    'CUSTOMERS.AFFECTED', 'TOTAL.CUSTOMERS', 'POPDEN_URBAN'
]

outages_df = outages_df[columns_to_keep].drop(0).reset_index(drop=True)
outages_df.head()

,YEAR,MONTH,U.S._STATE,NERC.REGION,CLIMATE.REGION,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CLIMATE.CATEGORY,OUTAGE.DURATION,CUSTOMERS.AFFECTED,TOTAL.CUSTOMERS,POPDEN_URBAN
0,2011.0,7.0,Minnesota,MRO,East North Central,"Friday, July 1, 2011",5:00:00 PM,"Sunday, July 3, 2011",8:00:00 PM,severe weather,normal,3060,70000.0,2595696.0,2279
1,2014.0,5.0,Minnesota,MRO,East North Central,"Sunday, May 11, 2014",6:38:00 PM,"Sunday, May 11, 2014",6:39:00 PM,intentional attack,normal,1,NaN,2640737.0,2279
2,2010.0,10.0,Minnesota,MRO,East North Central,"Tuesday, October 26, 2010",8:00:00 PM,"Thursday, October 28, 2010",10:00:00 PM,severe weather,cold,3000,70000.0,2586905.0,2279
3,2012.0,6.0,Minnesota,MRO,East North Central,"Tuesday, June 19, 2012",4:30:00 AM,"Wednesday, June 20, 2012",11:00:00 PM,severe weather,normal,2550,68200.0,2606813.0,2279
4,2015.0,7.0,Minnesota,MRO,East North Central,"Saturday, July 18, 2015",2:00:00 AM,"Sunday, July 19, 2015",7:00:00 AM,severe weather,warm,1740,250000.0,2673531.0,2279


In [3]:
outages_df['OUTAGE.START'] = pd.to_datetime(
    outages_df['OUTAGE.START.DATE'] + ' ' + outages_df['OUTAGE.START.TIME'],
    format='%A, %B %d, %Y %I:%M:%S %p'
)

outages_df['OUTAGE.RESTORATION'] = pd.to_datetime(
    outages_df['OUTAGE.RESTORATION.DATE'] + ' ' + outages_df['OUTAGE.RESTORATION.TIME'],
    format='%A, %B %d, %Y %I:%M:%S %p'
)

outages_df = outages_df.drop(columns=[
    'OUTAGE.START.DATE', 'OUTAGE.START.TIME',
    'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME'
])

`OUTAGE.DURATION` and other numeric-looking columns are cast to numeric, coercing bad values to `NaN`. Rows with `OUTAGE.DURATION == 0` are also set to `NaN`: a *major* outage lasting 0 minutes isn't physically meaningful, so a recorded 0 more likely reflects a data-entry artifact than a real event.

In [4]:
for col in ['OUTAGE.DURATION', 'CUSTOMERS.AFFECTED', 'MONTH', 'YEAR', 'POPDEN_URBAN']:
    outages_df[col] = pd.to_numeric(outages_df[col], errors='coerce')

outages_df['OUTAGE.DURATION'] = outages_df['OUTAGE.DURATION'].replace(0, np.nan)
outages_df['CAUSE.CATEGORY'] = outages_df['CAUSE.CATEGORY'].str.lower().str.strip()
outages_df.head()

,YEAR,MONTH,U.S._STATE,NERC.REGION,CLIMATE.REGION,CAUSE.CATEGORY,CLIMATE.CATEGORY,OUTAGE.DURATION,CUSTOMERS.AFFECTED,TOTAL.CUSTOMERS,POPDEN_URBAN,OUTAGE.START,OUTAGE.RESTORATION
0,2011.0,7.0,Minnesota,MRO,East North Central,severe weather,normal,3060.0,70000.0,2595696.0,2279.0,2011-07-01 17:00:00,2011-07-03 20:00:00
1,2014.0,5.0,Minnesota,MRO,East North Central,intentional attack,normal,1.0,NaN,2640737.0,2279.0,2014-05-11 18:38:00,2014-05-11 18:39:00
2,2010.0,10.0,Minnesota,MRO,East North Central,severe weather,cold,3000.0,70000.0,2586905.0,2279.0,2010-10-26 20:00:00,2010-10-28 22:00:00
3,2012.0,6.0,Minnesota,MRO,East North Central,severe weather,normal,2550.0,68200.0,2606813.0,2279.0,2012-06-19 04:30:00,2012-06-20 23:00:00
4,2015.0,7.0,Minnesota,MRO,East North Central,severe weather,warm,1740.0,250000.0,2673531.0,2279.0,2015-07-18 02:00:00,2015-07-19 07:00:00


### Univariate Analysis

In [5]:
outages_by_state = outages_df['U.S._STATE'].value_counts().reset_index()
outages_by_state.columns = ['STATE', 'OUTAGE_COUNT']

state_to_code = {state.name: state.abbr for state in states.STATES}
outages_by_state['STATE_CODE'] = outages_by_state['STATE'].map(state_to_code)

fig = px.choropleth(
    outages_by_state, locations='STATE_CODE', locationmode='USA-states',
    color='OUTAGE_COUNT', scope='usa', color_continuous_scale='sunset',
    title='Power Outages by U.S. State', hover_name='STATE',
    labels={'OUTAGE_COUNT': 'Outages'}
)
fig.update_layout(geo=dict(bgcolor='rgba(0,0,0,0)'), margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.write_html('assets/outages_map.html', full_html=True, include_plotlyjs='cdn')
fig.show()

In [6]:
outages_per_year = outages_df['YEAR'].value_counts().sort_index().reset_index()
outages_per_year.columns = ['Year', 'Number of Outages']

fig = px.line(outages_per_year, x='Year', y='Number of Outages', title='Power Outages Over Time')
fig.write_html('assets/outages_overtime.html', full_html=True, include_plotlyjs='cdn')
fig.show()

In [7]:
outages_by_region = outages_df['CLIMATE.REGION'].value_counts().reset_index()
outages_by_region.columns = ['Climate Region', 'Number of Outages']

fig = px.bar(outages_by_region, x='Climate Region', y='Number of Outages',
             title='Power Outages by Climate Region', color='Climate Region',
             text='Number of Outages')
fig.write_html('assets/outages_climate_region.html', full_html=True, include_plotlyjs='cdn')
fig.show()

### Bivariate Analysis

In [8]:
outages_clean = outages_df.dropna(subset=['CLIMATE.REGION', 'OUTAGE.DURATION']).copy()
ordered_regions = outages_clean.groupby('CLIMATE.REGION')['OUTAGE.DURATION'].median().sort_values().index
outages_clean['CLIMATE.REGION'] = pd.Categorical(outages_clean['CLIMATE.REGION'], categories=ordered_regions, ordered=True)

fig = px.box(outages_clean, x='CLIMATE.REGION', y='OUTAGE.DURATION',
             title='Outage Duration by Climate Region', color='CLIMATE.REGION')
fig.update_yaxes(tickformat=',', title='Outage Duration (minutes)')
fig.update_layout(showlegend=False)
fig.write_html('assets/outage_duration_climateregion.html', full_html=True, include_plotlyjs='cdn')
fig.show()

In [9]:
scatter_df = outages_df.dropna(subset=['OUTAGE.DURATION', 'CAUSE.CATEGORY'])

fig = px.scatter(scatter_df, x='CAUSE.CATEGORY', y='OUTAGE.DURATION',
                  title='Outage Duration vs Cause Category', color='CAUSE.CATEGORY',
                  opacity=0.6, height=600)
fig.update_yaxes(title='Outage Duration (minutes)')
fig.update_xaxes(title='Cause Category')
fig.update_layout(showlegend=False)
fig.write_html('assets/outage_duration_cause_cat.html', full_html=True, include_plotlyjs='cdn')
fig.show()

### Grouping and Aggregation

In [10]:
agg_summary = outages_df.groupby('CAUSE.CATEGORY').agg(
    **{'Avg Duration': ('OUTAGE.DURATION', 'mean'),
       'Median Duration': ('OUTAGE.DURATION', 'median'),
       'Max Duration': ('OUTAGE.DURATION', 'max'),
       'Number of Outages': ('YEAR', 'count')}
).reset_index().rename(columns={'CAUSE.CATEGORY': 'Cause Category'})

agg_summary = agg_summary.sort_values(by='Avg Duration', ascending=False).reset_index(drop=True)
agg_summary.to_html('assets/agg_summary_table.html', index=False)
agg_summary

,Cause Category,Avg Duration,Median Duration,Max Duration,Number of Outages
0,fuel supply emergency,13484.026316,3960.0,108653.0,51
1,severe weather,3899.709852,2464.0,49320.0,763
2,equipment failure,1850.555556,224.0,78377.0,60
3,public appeal,1468.449275,455.0,11867.0,69
4,system operability disruption,747.091667,222.0,23187.0,127
5,intentional attack,521.933735,92.5,21360.0,418
6,islanding,200.545455,77.5,1254.0,46


In [11]:
pivot = pd.pivot_table(
    outages_df, values='YEAR', index='CLIMATE.REGION', columns='CAUSE.CATEGORY', aggfunc='count'
)
pivot.to_html('assets/pivot_table_year.html')
pivot

CAUSE.CATEGORY,equipment failure,fuel supply emergency,intentional attack,islanding,public appeal,severe weather,system operability disruption
CLIMATE.REGION,,,,,,,
Central,7.0,4.0,38.0,3.0,2.0,135.0,11.0
East North Central,3.0,5.0,20.0,1.0,2.0,104.0,3.0
Northeast,5.0,14.0,135.0,1.0,4.0,176.0,15.0
Northwest,2.0,1.0,89.0,5.0,2.0,29.0,4.0
South,10.0,7.0,28.0,2.0,42.0,113.0,27.0
Southeast,5.0,NaN,9.0,NaN,5.0,118.0,16.0
Southwest,5.0,2.0,64.0,1.0,1.0,10.0,9.0
West,21.0,17.0,31.0,28.0,9.0,70.0,41.0
West North Central,1.0,1.0,4.0,5.0,2.0,4.0,NaN


## Step 3: Assessment of Missingness *(joint work)*

This section explores how outage durations differ depending on cause — specifically comparing *Severe Weather* events to all *Other Causes*. The goal is to check, visually, whether outages attributed to severe weather tend to last longer than those caused by other factors before formally testing it in Step 4.

In [12]:
others = outages_df[outages_df['CAUSE.CATEGORY'] != 'severe weather']
prop_other = others['CAUSE.CATEGORY'].value_counts(normalize=True).sort_values().reset_index()
prop_other.columns = ['Cause Category', 'Proportion']

fig = px.bar(prop_other, x='Proportion', y='Cause Category', orientation='h',
             text=prop_other['Proportion'].apply(lambda p: f'{p:.1%}'),
             title='Breakdown of Outage Causes (Excluding Severe Weather)')
fig.update_traces(marker_color='steelblue', textposition='outside')
fig.update_layout(xaxis_title='Proportion of Other Outages', yaxis_title='Cause Category',
                   yaxis=dict(categoryorder='total ascending'), template='ggplot2')
fig.write_html('assets/missingness_proportion.html', full_html=True, include_plotlyjs='cdn')
fig.show()

In [13]:
plot_df = outages_df.copy()
plot_df['DURATION_HOURS'] = plot_df['OUTAGE.DURATION'] / 60
plot_df = plot_df.dropna(subset=['DURATION_HOURS', 'CAUSE.CATEGORY'])
plot_df['Cause Group'] = np.where(plot_df['CAUSE.CATEGORY'] == 'severe weather', 'Severe Weather', 'Other Causes')

fig_box = px.box(plot_df, x='Cause Group', y='DURATION_HOURS', color='Cause Group',
                  points='outliers', notched=True, title='Outage Duration by Cause',
                  labels={'DURATION_HOURS': 'Outage Duration (hours)'})
fig_box.update_layout(showlegend=False)
fig_box.write_html('assets/missingness_box.html', full_html=True, include_plotlyjs='cdn')
fig_box.show()

fig_hist = px.histogram(plot_df, x='DURATION_HOURS', color='Cause Group', nbins=30,
                         barmode='overlay', title='Distribution of Outage Durations',
                         labels={'DURATION_HOURS': 'Outage Duration (hours)'})
fig_hist.update_traces(opacity=0.7)
fig_hist.write_html('assets/missingness_hist.html', full_html=True, include_plotlyjs='cdn')
fig_hist.show()

**Boxplot insight**: Severe Weather outages have a higher median and a wider IQR than Other Causes, with more extreme outliers.

**Histogram insight**: Severe Weather durations are right-skewed with a long tail past 500 hours, while Other Causes cluster more tightly around 20-40 hours. This is exactly the kind of skew that motivates the log-transform used in the modeling section below — a model that treats raw minutes as roughly normal will be dominated by these rare, very long outages.

## Step 4: Hypothesis Testing *(joint work)*

**Null Hypothesis (H₀):** Severe weather has no effect on the length of a power outage.

**Alternative Hypothesis (H₁):** Severe weather has an effect on the length of a power outage (either longer or shorter).

**Test statistic:** Difference in group means (Severe Weather − Other Causes).

**Significance level:** 0.05

In [14]:
severe = outages_df[outages_df['CAUSE.CATEGORY'] == 'severe weather']['OUTAGE.DURATION'].dropna()
non_severe = outages_df[outages_df['CAUSE.CATEGORY'] != 'severe weather']['OUTAGE.DURATION'].dropna()

observed_diff = severe.mean() - non_severe.mean()
print(f"Observed difference (Severe - Non Severe): {observed_diff:.2f} minutes")

Observed difference (Severe - Non Severe): 2399.86 minutes


In [15]:
np.random.seed(42)
combined = np.concatenate([severe, non_severe])
n_perm = 10000
perm_diffs = np.zeros(n_perm)

for i in range(n_perm):
    shuffled = np.random.permutation(combined)
    perm_diffs[i] = shuffled[:len(severe)].mean() - shuffled[len(severe):].mean()

p_val = (np.abs(perm_diffs) >= np.abs(observed_diff)).mean()

print(f"Observed difference: {observed_diff:.2f} minutes")
print(f"Permutation p-value: {p_val:.4f}")
print(f"Conclusion: {'REJECT' if p_val < 0.05 else 'DO NOT REJECT'} the null hypothesis")

Observed difference: 2399.86 minutes


Permutation p-value: 0.0000
Conclusion: REJECT the null hypothesis


In [16]:
perm_df = pd.DataFrame({'diffs': perm_diffs})
fig = px.histogram(perm_df, x='diffs', nbins=50, title=f'Permutation Test: p-value = {p_val:.4f}',
                    labels={'diffs': 'Difference in Means (Severe - Non Severe)'})
fig.add_vline(x=observed_diff, line_dash='dash', line_color='red',
              annotation_text=f'Observed Diff = {observed_diff:.1f}', annotation_position='top right')
fig.add_vline(x=-observed_diff, line_dash='dash', line_color='red',
              annotation_text=f'Observed Diff = -{observed_diff:.1f}', annotation_position='top right')
fig.write_html('assets/permutation_test_5.html', full_html=True, include_plotlyjs='cdn')
fig.show()

With a p-value of 0.0000 (0 of 10,000 permutations produced a difference at least as extreme as the observed one), we **reject the null hypothesis**: severe weather is associated with significantly longer outage durations than other causes.

## Step 5: Framing the Prediction Problem *(v2 — independent rework, sole credit)*

> Everything from here to the end of the notebook (Steps 5-8) is an independent rework of the original project's modeling section. The original v1 baseline/final linear regression models topped out at **R² = 0.045** on raw-minute outage duration. Rather than re-polish that approach, this version diagnoses *why* it stalled and rebuilds the pipeline around that diagnosis. See `README.md` for the full write-up and the original v1 numbers for comparison.

**Why the v1 model stalled:** `OUTAGE.DURATION` is massively right-skewed — most outages resolve in hours, but a long tail runs for weeks (see the histogram in Step 3). A linear model minimizing squared error on raw minutes is dominated by that handful of multi-week outages, and R² on the minutes scale is correspondingly unstable and uninformative. `U.S._STATE` (49 categories) also one-hot-encodes into a large, sparse block relative to the ~1,400 usable rows, which invites overfitting without adding much signal beyond what `CLIMATE.REGION`/`NERC.REGION` already capture.

**Two ways we address this:**
1. **Regression on log-duration.** Predict `log1p(OUTAGE.DURATION)` instead of raw minutes, using gradient boosting instead of linear regression to capture non-linear/interaction effects between cause, region, and season. We report **MAE in hours** (interpretable) rather than leading with RMSE/R² on the raw-minutes scale.
2. **Reframing as classification.** Predict a duration *bucket* (`<1 day`, `1-3 days`, `3+ days`) instead of an exact duration. This is more forgiving statistically and arguably more useful in practice — it's the kind of triage signal ("should we tell customers to expect same-day power, or plan for multi-day") a utility would actually act on.

**Features used** (all knowable at or shortly after the moment an outage begins — no post-hoc information about how the outage was ultimately resolved): `CAUSE.CATEGORY`, `CLIMATE.REGION`, `NERC.REGION`, `SEASON` (derived from `MONTH`), `MONTH`, `YEAR`, `START_HOUR` (derived from `OUTAGE.START`), `POPDEN_URBAN`.

**Evaluation metric:** MAE in hours as the headline number (interpretable, robust to the long tail), with R² reported on both the raw-minutes scale and the log scale it was optimized on — flagging the raw-minutes R² as unstable rather than treating it as the primary metric.

In [17]:
outages_df['START_HOUR'] = outages_df['OUTAGE.START'].dt.hour

season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
              3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer',
              9: 'Fall', 10: 'Fall', 11: 'Fall'}
outages_df['SEASON'] = outages_df['MONTH'].map(season_map)

features = ['CAUSE.CATEGORY', 'CLIMATE.REGION', 'NERC.REGION', 'SEASON',
            'MONTH', 'YEAR', 'START_HOUR', 'POPDEN_URBAN']
target = 'OUTAGE.DURATION'

model_df = outages_df.dropna(subset=features + [target]).copy()
print(f"Rows available for modeling: {len(model_df)}")

X = model_df[features]
y = model_df[target]
y_log = np.log1p(y)

X_train, X_test, y_train, y_test, y_log_train, y_log_test = train_test_split(
    X, y, y_log, test_size=0.2, random_state=42
)

categorical_cols = ['CAUSE.CATEGORY', 'CLIMATE.REGION', 'NERC.REGION', 'SEASON']
numerical_cols = ['MONTH', 'YEAR', 'START_HOUR', 'POPDEN_URBAN']

Rows available for modeling: 1393


## Step 6: Baseline Model *(v2)*

A trivial baseline for context: always predict the training-set mean duration. Any model we build should beat this by a meaningful margin, and we should be honest if it barely does.

In [18]:
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

print("=== Dummy (mean) baseline ===")
print(f"MAE:  {mean_absolute_error(y_test, pred_dummy) / 60:.2f} hours")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, pred_dummy)) / 60:.2f} hours")
print(f"R²:   {r2_score(y_test, pred_dummy):.4f}")

=== Dummy (mean) baseline ===
MAE:  53.41 hours
RMSE: 115.10 hours
R²:   -0.0009


A slightly stronger baseline, closer in spirit to the original v1 model: linear regression on the raw-minutes target, using the same feature set as the v2 model above (but without the log-transform or gradient boosting) so the *only* thing that changes going into Step 7 is the modeling approach, not the features.

In [19]:
baseline_lin = Pipeline([
    ('preprocessor', make_column_transformer(
        (OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        (StandardScaler(), numerical_cols)
    )),
    ('regressor', LinearRegression())
])
baseline_lin.fit(X_train, y_train)
pred_baseline = baseline_lin.predict(X_test)

print("=== Linear Regression baseline (raw minutes, v2 feature set) ===")
print(f"MAE:  {mean_absolute_error(y_test, pred_baseline) / 60:.2f} hours")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, pred_baseline)) / 60:.2f} hours")
print(f"R²:   {r2_score(y_test, pred_baseline):.4f}")

=== Linear Regression baseline (raw minutes, v2 feature set) ===
MAE:  43.36 hours
RMSE: 103.86 hours
R²:   0.1851


For reference, the original v1 project's final linear regression model (different feature set, including `U.S._STATE`) reported **MAE ≈ 2934.65 min (≈ 48.9 hrs)**, **RMSE ≈ 7422.66 min (≈ 123.7 hrs)**, **R² ≈ 0.0449** on raw minutes. (Its accompanying text claimed RMSE *decreased* from the v1 baseline's 7264 min — it actually increased; that's the sentence this rework corrects.)

## Step 7: Final Model — Gradient Boosting on Log-Duration *(v2)*

In [20]:
gb_pipeline = Pipeline([
    ('preprocessor', make_column_transformer(
        (OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('passthrough', numerical_cols)
    )),
    ('regressor', HistGradientBoostingRegressor(random_state=42, max_iter=300, learning_rate=0.05))
])

gb_pipeline.fit(X_train, y_log_train)
pred_log = gb_pipeline.predict(X_test)
pred_minutes = np.clip(np.expm1(pred_log), 0, None)

mae_hours = mean_absolute_error(y_test, pred_minutes) / 60
rmse_hours = np.sqrt(mean_squared_error(y_test, pred_minutes)) / 60
r2_minutes = r2_score(y_test, pred_minutes)
r2_log = r2_score(y_log_test, pred_log)

print("=== Final Model: HistGradientBoostingRegressor on log1p(duration) ===")
print(f"MAE:  {mae_hours:.2f} hours")
print(f"RMSE: {rmse_hours:.2f} hours")
print(f"R² (log scale, matches optimization target): {r2_log:.4f}")
print(f"R² (raw-minutes scale, unstable due to long tail): {r2_minutes:.4f}")

=== Final Model: HistGradientBoostingRegressor on log1p(duration) ===
MAE:  38.59 hours
RMSE: 110.37 hours
R² (log scale, matches optimization target): 0.5321
R² (raw-minutes scale, unstable due to long tail): 0.0798


**Reading these numbers honestly:** MAE improves from ~43.4 hours (linear baseline) to ~38.6 hours (gradient boosting + log target) — a real but modest gain, not a breakthrough. R² on the log scale (0.53) looks much better than R² on the raw-minutes scale (0.08), and that gap is the point: **the raw-minutes R² is dominated by a small number of multi-week outages and is not a reliable summary of fit for this target.** The log-scale R² is a fairer read of how well the model captures *relative* differences in duration (e.g., "this outage will likely take roughly twice as long as that one"), which is the more decision-useful framing anyway.

We do **not** claim this solves the duration-prediction problem. Outage duration substantially depends on information this dataset doesn't capture at outage onset — crew dispatch time, extent of physical damage, weather evolution during the event — so a real ceiling on predictability from these features alone is a legitimate, well-diagnosed finding, not just a modeling shortfall.

In [21]:
residuals_hours = (y_test - pred_minutes) / 60
resid_df = pd.DataFrame({'Residuals (hours)': residuals_hours})

fig = px.histogram(resid_df, x='Residuals (hours)', nbins=40, opacity=0.75,
                    color_discrete_sequence=['tomato'], title='Final Model Residuals (hours)')
fig.add_vline(x=0, line_width=2, line_dash='dash', line_color='black',
              annotation_text='Zero Error', annotation_position='top left')
fig.add_vline(x=residuals_hours.mean(), line_width=2, line_dash='dash', line_color='red',
              annotation_text=f'Mean Residual = {residuals_hours.mean():.1f}h', annotation_position='top right')
fig.update_layout(template='plotly_white')
fig.write_html('assets/final_model_v2_residuals.html', full_html=True, include_plotlyjs='cdn')
fig.show()

### Step 7b: Reframing as Classification — Duration Buckets

Predicting exact duration is a hard regression problem given these features. Reframing it as a 3-class classification problem — **`<1 day`**, **`1-3 days`**, **`3+ days`** — is both more statistically forgiving and more actionable: it mirrors the kind of triage message ("expect same-day restoration" vs. "plan for a multi-day outage") a utility would actually communicate to customers.

In [22]:
def bucket(minutes):
    days = minutes / (60 * 24)
    if days < 1:
        return '<1 day'
    elif days <= 3:
        return '1-3 days'
    return '3+ days'

y_bucket = y.apply(bucket)
y_bucket_train = y_bucket.loc[X_train.index]
y_bucket_test = y_bucket.loc[X_test.index]

print("Class distribution:")
print(y_bucket.value_counts(normalize=True).round(3))

Class distribution:
OUTAGE.DURATION
<1 day      0.582
1-3 days    0.248
3+ days     0.170
Name: proportion, dtype: float64


In [23]:
clf_pipeline = Pipeline([
    ('preprocessor', make_column_transformer(
        (OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('passthrough', numerical_cols)
    )),
    ('classifier', HistGradientBoostingClassifier(random_state=42, max_iter=300, learning_rate=0.05))
])

clf_pipeline.fit(X_train, y_bucket_train)
pred_bucket = clf_pipeline.predict(X_test)

accuracy = accuracy_score(y_bucket_test, pred_bucket)
f1_macro = f1_score(y_bucket_test, pred_bucket, average='macro')
majority_class = y_bucket_train.value_counts().idxmax()
majority_baseline_acc = (y_bucket_test == majority_class).mean()

print("=== Duration-Bucket Classifier ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 (macro): {f1_macro:.4f}")
print(f"Majority-class baseline ('{majority_class}'): {majority_baseline_acc:.4f}")
print()
print(classification_report(y_bucket_test, pred_bucket))

=== Duration-Bucket Classifier ===
Accuracy: 0.6810
F1 (macro): 0.6007
Majority-class baseline ('<1 day'): 0.5986

              precision    recall  f1-score   support

    1-3 days       0.40      0.46      0.43        63
     3+ days       0.61      0.51      0.56        49
      <1 day       0.82      0.81      0.82       167

    accuracy                           0.68       279
   macro avg       0.61      0.59      0.60       279
weighted avg       0.69      0.68      0.68       279



In [24]:
cm = confusion_matrix(y_bucket_test, pred_bucket, labels=['<1 day', '1-3 days', '3+ days'])
cm_df = pd.DataFrame(cm, index=['<1 day', '1-3 days', '3+ days'], columns=['<1 day', '1-3 days', '3+ days'])

fig = px.imshow(cm_df, text_auto=True, color_continuous_scale='blues',
                 title='Confusion Matrix: Duration Bucket Classifier',
                 labels=dict(x='Predicted', y='Actual', color='Count'))
fig.write_html('assets/bucket_confusion_matrix.html', full_html=True, include_plotlyjs='cdn')
fig.show()

The classifier beats the majority-class baseline by a clear margin and, per the classification report, does reasonably well distinguishing `<1 day` outages from longer ones — the practically important distinction for triage — while `1-3 days` vs `3+ days` is a harder boundary (fewer examples, less separable given these features).

## Step 8: Fairness Analysis *(v2)*

We check whether the final regression model (Step 7) is equally accurate across outage causes, comparing **Severe Weather** vs. **Equipment Failure**.

**Null Hypothesis (H₀):** The model is fair — RMSE is the same for both groups, and any observed difference is due to chance.

**Alternative Hypothesis (H₁):** The model is unfair — RMSE differs meaningfully between the two groups.

**Test statistic:** Difference in RMSE (Severe Weather − Equipment Failure), evaluated via permutation test.

In [25]:
fair_df = model_df.loc[X_test.index, ['CAUSE.CATEGORY']].copy()
fair_df['y_true'] = y_test.values
fair_df['y_pred'] = pred_minutes
fair_df = fair_df[fair_df['CAUSE.CATEGORY'].isin(['severe weather', 'equipment failure'])].copy()

print("Group sizes in test set:")
print(fair_df['CAUSE.CATEGORY'].value_counts())

Group sizes in test set:
CAUSE.CATEGORY
severe weather       151
equipment failure     12
Name: count, dtype: int64


**Caveat before looking at the numbers:** the test set has only a handful of Equipment Failure examples (a single-digit-to-low-teens count). RMSE for that group is estimated from very few points and will be noisy regardless of what the model does — any conclusion here should be read with that in mind, not as a confident fairness verdict.

In [26]:
def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

rmse_severe = rmse(
    fair_df[fair_df['CAUSE.CATEGORY'] == 'severe weather']['y_true'],
    fair_df[fair_df['CAUSE.CATEGORY'] == 'severe weather']['y_pred']
)
rmse_equipment = rmse(
    fair_df[fair_df['CAUSE.CATEGORY'] == 'equipment failure']['y_true'],
    fair_df[fair_df['CAUSE.CATEGORY'] == 'equipment failure']['y_pred']
)
observed_diff = rmse_severe - rmse_equipment

print(f"RMSE, Severe Weather:    {rmse_severe / 60:.2f} hours")
print(f"RMSE, Equipment Failure: {rmse_equipment / 60:.2f} hours")
print(f"Observed difference:     {observed_diff / 60:.2f} hours")

RMSE, Severe Weather:    62.20 hours
RMSE, Equipment Failure: 371.88 hours
Observed difference:     -309.68 hours


In [27]:
np.random.seed(42)
n_reps = 2000
labels = fair_df['CAUSE.CATEGORY'].values
diffs = np.zeros(n_reps)

for i in range(n_reps):
    shuffled = np.random.permutation(labels)
    mask_sw = shuffled == 'severe weather'
    mask_eq = shuffled == 'equipment failure'
    diffs[i] = rmse(fair_df['y_true'][mask_sw], fair_df['y_pred'][mask_sw]) - \
               rmse(fair_df['y_true'][mask_eq], fair_df['y_pred'][mask_eq])

p_value = (np.abs(diffs) >= np.abs(observed_diff)).mean()
print(f"Permutation p-value (two-sided, {n_reps} reps): {p_value:.4f}")

Permutation p-value (two-sided, 2000 reps): 0.0795


In [28]:
diffs_df = pd.DataFrame({'Diffs (hours)': diffs / 60})
fig = px.histogram(diffs_df, x='Diffs (hours)', nbins=30, opacity=0.75,
                    color_discrete_sequence=['coral'],
                    title='Permutation Test: RMSE Difference (Severe Weather - Equipment Failure)')
fig.add_vline(x=observed_diff / 60, line_width=2, line_dash='dash', line_color='red',
              annotation_text=f'Observed = {observed_diff/60:.1f}h', annotation_position='top right')
fig.update_layout(template='plotly_white')
fig.write_html('assets/fairness_analysis_v2.html', full_html=True, include_plotlyjs='cdn')
fig.show()

**Conclusion:** the permutation p-value is **0.0795** — above the 0.05 threshold, so we **fail to reject** the null hypothesis. Taken at face value, that says we don't have significant evidence the model is unfair across these two groups. But given only 12 Equipment Failure examples in the test set, this test has limited statistical power in either direction; a larger, more balanced sample across cause categories would be needed before drawing a confident fairness conclusion one way or the other.

## Summary

| Model | Target | MAE | R² | Notes |
|---|---|---|---|---|
| v1 (original, joint project) — Linear Regression | raw minutes | ≈ 48.9 hrs | 0.045 | original `U.S._STATE`-based feature set |
| v2 Baseline — Linear Regression | raw minutes | 43.4 hrs | 0.185 | same v2 feature set as Final Model, no log-transform/GB |
| v2 Final — Gradient Boosting | log1p(minutes) | 38.6 hrs | 0.53 (log scale) / 0.08 (raw-minutes) | headline model |
| v2 Classification — Gradient Boosting | duration bucket | — | accuracy 0.68 vs. 0.60 majority-class baseline; F1 (macro) 0.60 | more forgiving, more actionable framing |

The honest takeaway: outage duration is genuinely hard to predict from cause/region/season/time-of-day alone. Swapping the high-cardinality, low-signal `U.S._STATE` feature for `CLIMATE.REGION` + `SEASON`, and moving off raw-minutes linear regression, roughly quadruples R² (0.045 → 0.185) even before gradient boosting or the log-transform enter the picture — most of the v1 model's weakness was feature/target framing, not an unbeatable ceiling. Gradient boosting on log-duration pushes MAE down further (43.4 → 38.6 hours) and gives a much more honest R² story (0.53 on the scale it was actually optimized on, vs. an unstable 0.08 on raw minutes dominated by rare multi-week outages). None of this fully solves exact-duration prediction — real drivers like crew dispatch time and physical damage extent aren't in this dataset — but reframing the same information as a 3-bucket triage classifier (`<1 day` / `1-3 days` / `3+ days`) turns that same ceiling into a usable, well above-baseline decision signal for utilities.